# RQ7 decision support pipeline

**Research question:** How can causal risk scores and counterfactual recommendations be embedded into an institutional decision-support pipeline to prioritize at-risk students while reducing false or non-actionable alerts?

This notebook is designed for Kaggle execution and saves all result tables as CSV files and figures as PDF files under `/kaggle/working/results/rq7_decision_support_pipeline`.

In [1]:
import os, sys, json, math, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path('/kaggle/input/datasets/kimdaligermany/seoyeon-thesis-src')
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.config import ExperimentConfig
from src.data_utils import (load_generic_education_dataset,
                             build_weekly_timelines, derive_targets,
                             cumulative_snapshot, make_sequence_tensor)
from src.feature_engineering import numeric_feature_columns, make_preprocessor
from src.models import (fit_dual_tabular_model, predict_dual_tabular_model,
                        LSTMClassifier, TransformerClassifier,
                        MultiTaskCausalNet, train_torch_model,
                        predict_torch_multitask)
from src.evaluation import (classification_summary, summarise_dual_task,
                             topk_alert_precision, expected_calibration_error)
from src.causal_utils import (correlation_dag, direct_indirect_effects,
                               bootstrap_edge_stability)
from src.counterfactual_utils import (generate_simple_counterfactuals,
                                      generate_simple_counterfactuals_df,
                                      evaluate_counterfactuals,
                                      segment_joint_risk)
from src.plotting_utils import lineplot, scatterplot, heatmap, barplot

CFG = ExperimentConfig()
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'
CFG.dataset_name = "rq7_decision_support_pipeline"
OUT = CFG.output_dir
print("Output directory:", OUT)

Output directory: /kaggle/working/results/rq7_decision_support_pipeline


In [2]:
# Load raw logs, normalize them into a weekly timeline, and derive labels.
raw_df    = load_generic_education_dataset(DATASET_PATH,
                                           dataset_name=CFG.dataset_name)
weekly_df = build_weekly_timelines(raw_df)
weekly_df = derive_targets(weekly_df)
weekly_df.head()

,student_id,week,code_module,code_presentation,sum_click,n_activities,n_submissions,mean_score,late_rate,gender,age_band,imd_band,highest_education,num_of_prev_attempts,studied_credits,dropout,failure,final_result
0,6516,0,AAA,2014J,485,89,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
1,6516,1,AAA,2014J,42,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
2,6516,2,AAA,2014J,79,20,1.0,0.6,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
3,6516,3,AAA,2014J,193,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
4,6516,4,AAA,2014J,69,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass


## RQ7 workflow

This notebook embeds causal risk scores and counterfactual recommendations into a capacity-aware decision-support pipeline.

In [3]:
snap         = cumulative_snapshot(weekly_df, up_to_week=max(CFG.early_weeks))
feature_cols = numeric_feature_columns(snap)
X            = snap[feature_cols].fillna(0)
y_drop       = snap["dropout"].astype(int).values
y_fail       = snap["failure"].astype(int).values

X_tr, X_te, yd_tr, yd_te, yf_tr, yf_te = train_test_split(
    X, y_drop, y_fail,
    test_size=0.25, random_state=CFG.random_state, stratify=y_drop)

prep  = make_preprocessor()
X_trp = prep.fit_transform(X_tr)
X_tep = prep.transform(X_te)

X_tr_seq = np.repeat(
    X_trp[:, None, :], CFG.seq_max_len, axis=1).astype(np.float32)
X_te_seq = np.repeat(
    X_tep[:, None, :], CFG.seq_max_len, axis=1).astype(np.float32)
ytr      = np.vstack([yd_tr, yf_tr]).T.astype(np.float32)

model  = MultiTaskCausalNet(
    input_dim=X_tr_seq.shape[-1], hidden_dim=CFG.hidden_dim)
model  = train_torch_model(
    model, X_tr_seq, ytr,
    epochs=10, lr=CFG.learning_rate, batch_size=CFG.batch_size,
    causal_targets=X_tr_seq.mean(axis=(1, 2)), causal_lambda=0.12)
p_drop, p_fail = predict_torch_multitask(model, X_te_seq)

decision_df = X_te.reset_index(drop=True).copy()
decision_df["student_id"]   = (snap.loc[X_te.index, "student_id"].values
                                if "student_id" in snap.columns
                                else np.arange(len(X_te)))
decision_df["dropout_risk"] = p_drop
decision_df["failure_risk"] = p_fail
decision_df["overall_risk"] = (0.5 * decision_df["dropout_risk"]
                               + 0.5 * decision_df["failure_risk"])

cf_df = generate_simple_counterfactuals_df(
    decision_df, risk_col="overall_risk",
    top_n=min(200, len(decision_df)))

if not cf_df.empty:
    best_cf = (cf_df.sort_values("risk_reduction", ascending=False)
                    .drop_duplicates("student_id")
               [["student_id", "recommended_change",
                 "actionability", "revised_risk", "risk_reduction"]])
    decision_df = decision_df.merge(best_cf, on="student_id", how="left")
else:
    decision_df["recommended_change"] = "No action"
    decision_df["actionability"]      = 0.3
    decision_df["revised_risk"]       = decision_df["overall_risk"]
    decision_df["risk_reduction"]     = 0.0

decision_df["actionability"]      = decision_df["actionability"].fillna(0.3)
decision_df["risk_reduction"]     = decision_df["risk_reduction"].fillna(0.0)
decision_df["causal_actionable_score"] = (
    0.65 * decision_df["overall_risk"]
    + 0.35 * decision_df["actionability"])

print("decision_df shape:", decision_df.shape)
decision_df.head()

decision_df shape: (6507, 16)


,sum_click,n_activities,n_submissions,mean_score,late_rate,num_of_prev_attempts,studied_credits,student_id,dropout_risk,failure_risk,overall_risk,recommended_change,actionability,revised_risk,risk_reduction,causal_actionable_score
0,3.000,2.000,0.000000,0.000000,0.000000,1.0,60.0,496315,0.428158,0.508798,0.468478,NaN,0.3,NaN,0.0,0.409511
1,356.000,137.000,0.000000,0.000000,0.000000,0.0,60.0,688442,0.928237,0.076420,0.502329,NaN,0.3,NaN,0.0,0.431514
2,10.200,6.800,0.000000,0.000000,0.000000,0.0,30.0,596307,0.192161,0.379788,0.285974,NaN,0.3,NaN,0.0,0.290883
3,41.125,14.875,0.375000,0.315000,0.125000,0.0,120.0,587923,0.109216,0.112621,0.110919,NaN,0.3,NaN,0.0,0.177097
4,50.500,16.250,0.166667,0.156667,0.083333,0.0,60.0,503477,0.130395,0.146291,0.138343,NaN,0.3,NaN,0.0,0.194923


In [4]:
# Figure 7.1 simple pipeline diagram
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.axis("off")
steps = ["LMS logs", "Weekly timelines", "Temporal model",
         "Causal reasoning", "Risk estimation",
         "Counterfactual recourse", "Advisor triage"]
x_positions = np.linspace(0.07, 0.93, len(steps))

for x, step in zip(x_positions, steps):
    ax.text(x, 0.5, step, ha="center", va="center",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4",
                      facecolor="#EFF6FF",
                      edgecolor="#2563EB", linewidth=1.5))

for i in range(len(steps) - 1):
    ax.annotate("",
                xy=(x_positions[i+1] - 0.055, 0.5),
                xytext=(x_positions[i] + 0.055, 0.5),
                arrowprops=dict(arrowstyle="->", lw=1.5,
                                color="#2563EB"))

plt.title("Figure 7.1 Institutional decision-support pipeline",
          fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(
    str(OUT / "figure_7_1_institutional_decision_support_pipeline.pdf"),
    bbox_inches="tight", facecolor="white")
plt.close()
print("Figure 7.1 saved.")

Figure 7.1 saved.


In [5]:
capacity   = min(50, len(decision_df))
prob_rank  = (decision_df
              .sort_values("overall_risk", ascending=False)
              .head(capacity).copy())
causal_rank = (decision_df
               .sort_values("causal_actionable_score", ascending=False)
               .head(capacity).copy())

triage_df = pd.DataFrame([
    {
        "policy":               "Probability-only ranking",
        "students_contacted":   len(prob_rank),
        "true_at_risk_reached": int((prob_rank["overall_risk"] >= 0.5).sum()),
        "actionable_cases":     int((prob_rank["actionability"] >= 0.7).sum()),
        "false_non_actionable": int(
            ((prob_rank["overall_risk"] < 0.5)
             | (prob_rank["actionability"] < 0.5)).sum()),
        "net_utility":          round(
            0.7 * float((prob_rank["overall_risk"] >= 0.5).mean())
            + 0.3 * float(prob_rank["actionability"].mean()), 4),
    },
    {
        "policy":               "Causal-actionable ranking",
        "students_contacted":   len(causal_rank),
        "true_at_risk_reached": int((causal_rank["overall_risk"] >= 0.5).sum()),
        "actionable_cases":     int((causal_rank["actionability"] >= 0.7).sum()),
        "false_non_actionable": int(
            ((causal_rank["overall_risk"] < 0.5)
             | (causal_rank["actionability"] < 0.5)).sum()),
        "net_utility":          round(
            0.7 * float((causal_rank["overall_risk"] >= 0.5).mean())
            + 0.3 * float(causal_rank["actionability"].mean()), 4),
    },
])
triage_df.to_csv(
    OUT / "table_7_1_capacity_constrained_triage_outcomes.csv", index=False)
triage_df

,policy,students_contacted,true_at_risk_reached,actionable_cases,false_non_actionable,net_utility
0,Probability-only ranking,50,50,47,1,0.9668
1,Causal-actionable ranking,50,50,50,0,0.9972


In [6]:
# Figure 7.2 Workload-benefit decision curve
curve_rows = []
for cap in [10, 20, 30, 40, 50, 75, 100]:
    cap = min(cap, len(decision_df))
    pr  = decision_df.sort_values(
        "overall_risk", ascending=False).head(cap)
    cr  = decision_df.sort_values(
        "causal_actionable_score", ascending=False).head(cap)

    curve_rows.append({
        "capacity": cap,
        "policy":   "Probability-only ranking",
        "net_benefit": round(
            0.7 * float((pr["overall_risk"] >= 0.5).mean())
            + 0.3 * float(pr["actionability"].mean()), 4),
    })
    curve_rows.append({
        "capacity": cap,
        "policy":   "Causal-actionable ranking",
        "net_benefit": round(
            0.7 * float((cr["overall_risk"] >= 0.5).mean())
            + 0.3 * float(cr["actionability"].mean()), 4),
    })

curve_df = pd.DataFrame(curve_rows)
curve_df.to_csv(
    OUT / "table_7_2_workload_benefit_curve.csv", index=False)

lineplot(curve_df,
         x="capacity", y="net_benefit", hue="policy",
         title="Figure 7.2 Workload benefit decision curve",
         xlabel="Students contacted",
         ylabel="Net benefit",
         path=str(OUT / "figure_7_2_workload_benefit_decision_curve.pdf"))
curve_df.head()

,capacity,policy,net_benefit
0,10,Probability-only ranking,0.9773
1,10,Causal-actionable ranking,0.9960
2,20,Probability-only ranking,0.9714
3,20,Causal-actionable ranking,0.9947
4,30,Probability-only ranking,0.9633


In [7]:
# Fairness and actionability audit using pseudo-groups if no subgroup variable exists.
audit_df = decision_df.copy()

if "group" not in audit_df.columns:
    audit_df["group"] = pd.qcut(
        audit_df["student_id"].rank(method="first"),
        q=4,
        labels=["Group A", "Group B", "Group C", "Group D"]
    ).astype(str)

fairness = (
    audit_df.groupby("group")
            .agg(
                alert_precision     =("overall_risk",
                                      lambda s: round(float((s >= 0.5).mean()), 4)),
                recommendation_burden=("actionability",
                                       lambda s: round(float((1.0 - s).mean() + 1), 4)),
                actionability_score =("actionability",
                                      lambda s: round(float(s.mean()), 4)),
            )
            .reset_index()
)
fairness["flag"] = np.where(
    fairness["actionability_score"]
    < fairness["actionability_score"].mean() - 0.05,
    "Monitor", "Within tolerance")

fairness.to_csv(
    OUT / "table_7_3_fairness_and_actionability_audit.csv", index=False)
fairness

,group,alert_precision,recommendation_burden,actionability_score,flag
0,Group A,0.0738,1.6802,0.3198,Within tolerance
1,Group B,0.0768,1.6781,0.3219,Within tolerance
2,Group C,0.0449,1.6846,0.3154,Within tolerance
3,Group D,0.0430,1.6862,0.3138,Within tolerance
